[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivanvykopal/nlp-kinit-2026/blob/main/examples/peft/lora.ipynb)

# LoRA (PEFT) on Tower of Hanoi

Full fine-tuning rewrites all 135M weights, and at real model sizes that
gets expensive: the optimizer alone keeps roughly a model's worth of extra
state, and every task you adapt to leaves you another full copy of the
weights to store. LoRA — Low-Rank Adaptation, a **parameter-efficient
fine-tuning** (PEFT) method — freezes the original weights and trains a
small correction beside them instead. The theory section below works
through how.

The data, the objective and the step budget are the FFT notebook's, so
this notebook is a clean comparison of *what gets trained*. Watch two
numbers: the fraction of parameters that are trainable, which should be
tiny, and the held-out score, which should land in the same region as full
fine-tuning. Both methods are fitting the same imperfect demonstrations,
so both hit the same ceiling — parameter efficiency does not buy data
quality.

Runtime: ~7 minutes on a free Colab T4.

## 0. Setup

Install the dependencies (minimum versions these notebooks were validated against).

In [ ]:
# Colab preinstalls an old torchao (0.10.0) that the transformers below refuses
# to import alongside ("incompatible version of torchao"). These notebooks do
# no torchao quantization, so drop it instead of a torch-coupled upgrade.
get_ipython().system('pip uninstall -y torchao')
get_ipython().system('pip install -q "transformers>=4.56,<5" "trl>=0.24" "peft>=0.14" "datasets>=3.0" accelerate matplotlib pandas')

In [ ]:
get_ipython().system('git clone --depth 1 https://github.com/ivanvykopal/nlp-kinit-2026')
import sys; sys.path.insert(0, "nlp-kinit-2026")

from lab import evaluation, generation, plotting, report

## Shared code

The cells below define the constants (`config.py`) and the Tower of Hanoi
task logic (`tasks/hanoi.py`) directly in the notebook's own namespace —
run them once from the top; everything after uses these names without any
`import`. Generic infrastructure (batched generation, evaluation,
reporting, plotting) is *imported for real* in the cell above, from
`lab/`, which never mentions Hanoi by name.


In [ ]:
"""Generic constants shared by every notebook: model, dirs, token/training budgets.

No task-shaped constants here — those live in the task module (see
`tasks/hanoi.py`'s `PROBE_GROUP_VALUES`).
"""
from pathlib import Path

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
SEED = 42

Every notebook reads the same dataset from the HuggingFace Hub. It has five
named splits: `train` (answers to imitate, one in five deliberately wrong),
`grpo_train` (puzzles only, no answers — the split name predates our rename
to GSPO), `heldout` (the number we care about), `train_instances` (a
memorisation check) and `extrapolation` (bigger puzzles than any in
training).

In [ ]:
DATASET_REPO = "ivykopal/nlp-kinit-2026-hanoi"

Notebooks run from different working directories — `examples/rl/` locally,
`/content` on Colab — so a bare `Path("results")` would point somewhere
different in each one, and the GSPO notebook would fail to find the
checkpoint the FFT notebook saved. This walks up from the current directory
to find the one folder holding both `lab/` and `config.py`.

In [ ]:
def _repo_root() -> Path:
    import sys
    candidates = [Path.cwd(), *Path.cwd().parents]
    candidates += [Path(p) for p in sys.path if p]
    for c in candidates:
        if c.is_dir() and (c / "lab").is_dir() and (c / "config.py").is_file():
            return c
    return Path.cwd()  # fallback: original <cwd>/results behaviour


RESULTS_DIR = _repo_root() / "results"

How many tokens we let the model read and write. `MAX_SEQ_LENGTH` caps
training examples; `EVAL_MAX_NEW_TOKENS` has to be generous enough for a
31-move answer, or a correct solution gets cut off and scored as a failure.

In [ ]:
MAX_SEQ_LENGTH = 384
EVAL_MAX_NEW_TOKENS = 576
EVAL_BATCH_SIZE = 16

### BUDGETS: supervised fine-tuning

FFT and LoRA train on the same schedule and only differ in learning rate —
LoRA's update is confined to a low-rank subspace, so it needs a larger step
size to move as far as full fine-tuning does in the same number of steps.

In [ ]:
SFT_MAX_STEPS = 300
SFT_BATCH_SIZE = 4
SFT_GRAD_ACCUM = 4
FFT_LEARNING_RATE = 5e-5
LORA_LEARNING_RATE = 2e-4

## Task: Tower of Hanoi

This module is the one place that knows what "Tower of Hanoi" means:
how to parse a model's answer, how to check whether a sequence of moves
actually solves the puzzle, how to turn a dataset row into a training
example, and how to reward a rollout. Everything in `lab/` is generic —
it never mentions a peg or a disk by name — and depends only on the
functions below.

Swapping this tutorial to a different verifiable task (Sudoku, a graph
coloring problem, anything with a checker) means writing a new module
shaped like this one; `lab/` would not change at all.

A move is a plain `(source, target)` tuple of peg names — `("A", "C")`
reads as "move the top disk of A onto C". No `Move` class: a tuple is
already immutable, printable, and comparable, and one regex below is the
entire move syntax.

In [ ]:
"""Tower of Hanoi: parsing, solving, replaying, scoring, and reward."""
from __future__ import annotations

import random
import re
from typing import List

PEG_NAMES = ["A", "B", "C"]

# The whole move syntax: `A->C`, with optional spaces around the arrow.
MOVE_RE = re.compile(r"([{pegs}])\s*->\s*([{pegs}])".format(pegs="".join(PEG_NAMES)))

Two tiny helpers turn a move tuple into the text form the model reads and
writes, and back — everything else in this module works with `(source,
target)` tuples, so the text form only exists at the model boundary.

In [ ]:
def move_to_text(move) -> str:
    return "{}->{}".format(*move)


def moves_to_text(moves) -> str:
    return "\n".join(move_to_text(m) for m in moves)

Turning model output back into moves needs two different levels of
strictness. `parse_output` is used for scoring a whole flat solution: it
walks every line and keeps a strict per-line count, so junk is counted
rather than silently dropped.

In [ ]:
def parse_output(text: str):
    """Parse a whole completion into moves.

    Returns `(moves, n_lines, n_unparsed)`: every non-empty line is either a
    move or counted as unparseable — junk is counted, never silently
    dropped, so the caller can tell a clean list of moves from noisy output.
    """
    moves, n_lines, n_unparsed = [], 0, 0
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        n_lines += 1
        match = MOVE_RE.fullmatch(line)
        if match:
            moves.append(match.groups())
        else:
            n_unparsed += 1
    return moves, n_lines, n_unparsed

A reference solver, used two ways below: to build the optimal
demonstrations the SFT notebooks train on, and to know the optimal move
count for scoring — `2**n - 1`, the closed form for this recursion.

In [ ]:
def solve_hanoi(n_disks, source, auxiliary, target) -> List[tuple]:
    """The unique optimal solution: move n-1 aside, move the bottom disk, move them back."""
    if n_disks == 0:
        return []
    return (
        solve_hanoi(n_disks - 1, source, target, auxiliary)
        + [(source, target)]
        + solve_hanoi(n_disks - 1, auxiliary, source, target)
    )


def optimal_move_count(n_disks: int) -> int:
    return 2 ** n_disks - 1

### The verifier

There is no environment *class* here, and no simulator either. A board is
just a `dict` of `{peg: [disks]}`, top disk last in the list; `apply_move`
is the single stepping primitive (used by the flat replay below *and* by
the per-step formulation further down); and `replay` folds it over a list
of moves. An illegal move ends the attempt — whatever follows it is not
played, so it cannot count for anything.

In [ ]:
def new_pegs(n_disks, source, auxiliary, target) -> dict:
    """All disks stacked on `source`; the other two pegs start empty."""
    return {source: list(range(n_disks, 0, -1)), auxiliary: [], target: []}


def is_legal(pegs: dict, move) -> bool:
    """Legal iff it moves the top disk of one peg onto a bigger (or empty) peg."""
    source, target = move
    if source == target or not pegs[source]:
        return False
    return not pegs[target] or pegs[source][-1] < pegs[target][-1]

`apply_move` is the single stepping primitive — used by the flat `replay`
below.

In [ ]:
def apply_move(pegs: dict, move) -> "tuple[dict, bool]":
    """Legality-checked step. Returns (new_pegs, legal); `pegs` is never mutated."""
    if not is_legal(pegs, move):
        return pegs, False
    source, target = move
    stepped = {peg: list(stack) for peg, stack in pegs.items()}
    stepped[target].append(stepped[source].pop())
    return stepped, True


def is_solved(pegs: dict, n_disks: int, target: str) -> bool:
    return len(pegs[target]) == n_disks

`replay` folds `apply_move` over a whole list of moves, for scoring a flat
solution. An illegal move ends the attempt — whatever follows it is not
played, so it cannot count for anything.

In [ ]:
def replay(moves, n_disks, source, auxiliary, target) -> "tuple[dict, bool]":
    """Play `moves` from the start, stopping at the first illegal move.

    Returns the board it reached and whether an illegal move ended it.
    """
    pegs = new_pegs(n_disks, source, auxiliary, target)
    for move in moves:
        pegs, legal = apply_move(pegs, move)
        if not legal:
            return pegs, True
    return pegs, False

### Loading the dataset

The dataset lives on the HuggingFace Hub as five named splits — `train`,
`grpo_train`, `heldout`, `train_instances`, `extrapolation`. One row looks
like this:

```json
{
  "prompt": "...", "n_disks": 3, "source": "A", "auxiliary": "B", "target": "C",
  "target_response": "...", "optimal_response": "...",
  "corrupted": false, "corruption": null
}
```

In [ ]:
def load_split(name: str, repo_id: str):
    """Load one named split of the dataset from the HuggingFace Hub.

    Returns a `datasets.Dataset` — already iterable/indexable like a list of
    dicts (for `sum(r["corrupted"] for r in rows)`-style inspection) and
    already has `.map()` (for building the SFT/GSPO training format), so
    there's no separate "plain rows" vs. "Dataset" loading step.
    """
    from datasets import load_dataset

    return load_dataset(repo_id, split=name)

### Grouping for the by-group breakdown and the mid-training probe

Hanoi's natural grouping is disk count. A different task might group by
difficulty, grid size, or not at all — `lab.evaluation`/`lab.probe` accept
`group_key=None` and simply skip the breakdown when a task has nothing to
group by.

In [ ]:
def PROBE_GROUP_KEY(sample: dict) -> int:
    return sample["n_disks"]


PROBE_GROUP_VALUES = [4]

### `compute_stats`: what a model's answer scores as

`compute_stats` is what evaluation reports, and it deliberately reports
only three things: **did it solve the puzzle**, **how many moves it wrote
against the 2**n - 1 optimum**, and **was there anything in the output
that was not a move**. Nothing else — every extra field is one more column
to explain and one more thing that can quietly disagree with `solved`.

`"solved"` (a bool) is the only field `lab/` requires; the rest is
free-form, and `lab.evaluation.aggregate` averages whatever numeric/bool
fields it finds without needing to know their names.

In [ ]:
def compute_stats(prediction: str, sample: dict) -> dict:
    """Replay one completion and report what evaluation shows."""
    moves, n_lines, n_unparsed = parse_output(prediction)
    pegs, _illegal = replay(moves, sample["n_disks"], sample["source"],
                            sample["auxiliary"], sample["target"])
    return {
        "solved": is_solved(pegs, sample["n_disks"], sample["target"]),
        "total_moves": len(moves),
        "optimal_moves": optimal_move_count(sample["n_disks"]),
        "unparsed": n_unparsed > 0,
    }

### From dataset row to training example

Three small functions turn one dataset row into what a trainer expects: a
prompt, and — for the two supervised methods — the answer to train on.

In [ ]:
DEFAULT_SYSTEM_PROMPT = (
    "You are an expert algorithmic problem solver. "
    "Solve the Tower of Hanoi puzzle optimally. "
    "Return ONLY one move per line in the format 'A->C'. "
    "Do not provide any explanation."
)

`to_chat_prompt` wraps `DEFAULT_SYSTEM_PROMPT` and the puzzle's own prompt
text into the system+user half of a conversation. `to_sft_format` adds the
assistant turn FFT/LoRA train on.

In [ ]:
def to_chat_prompt(sample: dict) -> List[dict]:
    """The prompt half of the conversation (system + user)."""
    return [
        {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
        {"role": "user", "content": sample["prompt"]},
    ]


def to_sft_format(sample: dict) -> dict:
    """Prompt/completion format for SFTTrainer (loss on the answer only)."""
    return {
        "prompt": to_chat_prompt(sample),
        "completion": [{"role": "assistant", "content": sample["target_response"]}],
    }

## What fine-tuning actually changes

Start with the problem. The model we just downloaded is good at exactly
one thing: given some text, guess which token comes next. Nobody taught
it Tower of Hanoi. Ask it for a solution and you get something that
*looks* like an answer — confident, fluent, wrong.

**Fine-tuning** is how we fix that, and it is worth being precise about
what it does and does not do. It does not add a new skill from nothing.
It shifts *which* continuations the model finds likely, by showing it
examples of the input/output pairs we want. Supervised fine-tuning (SFT)
is the plainest version: collect prompts with the answers you wish the
model had given, and train it to give those answers.

Concretely, we minimise the negative log-likelihood of the answer tokens:

$$
\mathcal{L}(\theta) = -\sum_{t} \log p_\theta\big(y_t \mid x,\, y_{<t}\big)
$$

Read that piece by piece. $x$ is the prompt — here, a Tower of Hanoi
puzzle. $y$ is the answer we want, and $y_t$ is its $t$-th token. $\theta$
stands for every weight in the model, the things training is allowed to
change. Each term in the sum asks one question: given the puzzle and the
answer so far, how much probability did the model put on the *correct*
next token? Training pushes those probabilities up, which is the same as
pushing the loss down.

Now notice what the sum runs over: **answer tokens only**. That is what
`completion_only_loss=True` does in the config further down. We want the
model to get better at producing answers, not at predicting puzzles we
are going to type in ourselves.

![Loss is computed on answer tokens only](https://raw.githubusercontent.com/ivanvykopal/nlp-kinit-2026/main/images/sft_loss_mask.png)

*The prompt tokens are context; only the answer tokens contribute to the
loss. (Shown as whole words for readability; a real tokenizer splits many
of them into smaller subword pieces, but the masking works the same way —
by token, not by word.)*

This objective has a ceiling built into it, and the whole tutorial hinges
on seeing that ceiling. The loss is smallest when the model reproduces
the demonstrations — *all* of them, including the bad ones. One in five
answers in our training data has a deliberately broken move, so a model
that fits this data perfectly has also learned to make broken moves at
some rate. In a 31-move solution, where one illegal move invalidates
everything after it, that rate is fatal.

There are two ways out, and this tutorial covers both: score the model's
own attempts against a verifier instead of copying answers (the GSPO
notebook), or ask for one move at a time, so a mistake cannot compound
(the per-step notebooks).

## Adapting a model without touching its weights

Full fine-tuning updates every weight in the model. That is fine at 135M
parameters, but it has two costs that grow with the model: the optimizer
keeps state roughly as large as the model itself, and every task you
adapt to leaves you with a whole new copy of the weights to store and
serve.

**LoRA** (Low-Rank Adaptation) asks a cheaper question. Instead of
learning a new weight matrix, can we learn a small *correction* to the
one we already have? Freeze the pretrained weight $W \in \mathbb{R}^{d
\times k}$ and train two thin matrices next to it:

$$
W' = W + \frac{\alpha}{r}\, B A,
\qquad B \in \mathbb{R}^{d \times r},\;
A \in \mathbb{R}^{r \times k},\;
r \ll \min(d, k)
$$

Only $A$ and $B$ receive gradients; $W$ never changes. Because $r$ is
small — 16 in this notebook, against a hidden size in the hundreds — $A$
and $B$ together hold a tiny fraction of the parameters $W$ does.

Two knobs, and what they mean:

* the **rank** $r$ is how much the correction is allowed to express.
  Bigger $r$ means more capacity and more trainable parameters.
* the **scaling** $\alpha$ is how loudly the correction speaks relative
  to the frozen weights. Dividing by $r$ keeps that volume roughly
  constant when you change the rank, which is why you will see people
  tune $r$ and leave $\alpha$ alone. We use $r = 16$, $\alpha = 32$, so
  $\alpha/r = 2$.

![Frozen W beside the trainable low-rank B, A path](https://raw.githubusercontent.com/ivanvykopal/nlp-kinit-2026/main/images/lora_decomposition.png)

*W never receives a gradient; only the thin B and A matrices are trained, then scaled by alpha/r.*

One detail worth pausing on: $A$ starts random and $B$ starts at zero, so
$BA = 0$ at step 0 and the adapted model is *exactly* the original one.
Training begins from the pretrained behaviour rather than from noise —
which is also why a LoRA model evaluated before training behaves exactly
like the base model it wraps.

We attach adapters to seven projections in every layer: the four in
attention (`q_proj`, `k_proj`, `v_proj`, `o_proj`) and the three in the
feed-forward block (`gate_proj`, `up_proj`, `down_proj`). Attention-only
is the recipe from the original paper; including the feed-forward
projections costs a few more parameters, usually helps, and is the common
default today.

Finally, the reason LoRA is easy to like in production: $W + \frac{\alpha}{r}BA$
is just a matrix, so you can compute it once after training and ship an
ordinary model. LoRA costs nothing at inference time — unless you *want*
to keep the adapters separate and swap them per task, which you can.

We count the trainable parameters once the adapters are attached, a few
cells down.

## 1. The same shared dataset as the FFT notebook

Same splits, same Hub repo, loaded the same way. The only reason to print
the corruption count again is to confirm you are training on the identical
data the FFT notebook saw — one answer in five with a broken move.

In [ ]:
train_rows = load_split("train", DATASET_REPO)
eval_splits = {
    "heldout": load_split("heldout", DATASET_REPO),
    "train_instances": load_split("train_instances", DATASET_REPO),
    "extrapolation": load_split("extrapolation", DATASET_REPO),
}
corrupted = sum(r["corrupted"] for r in train_rows)
print(f"train: {len(train_rows)} rows, {corrupted} corrupted ({corrupted / len(train_rows):.0%})")

## 2. Base model plus adapters

The theory above explains what the adapters are; this cell is about the
three choices you have to make. **Rank** `r=16` is the usual starting
point for a task like this: large enough to express a useful correction,
small enough that the parameter count stays negligible. **`lora_alpha=32`**
gives `alpha/r = 2`, so raising the rank later does not also make the
correction louder. And the **seven target modules** are the four attention
projections plus the three feed-forward ones, in every layer — everything
except the embeddings and the LM head, which are the two biggest matrices
in a 135M model and the ones least worth adapting for a task this narrow.

In [ ]:
import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import SFTConfig, SFTTrainer

METHOD = "LoRA"
OUTPUT_DIR = RESULTS_DIR / "lora_model"
set_seed(SEED)
print("CUDA available:", torch.cuda.is_available())

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32)
model.config.use_cache = False
model.to("cuda" if torch.cuda.is_available() else "cpu")
print(f"model on: {next(model.parameters()).device}")

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

### How much are we actually training?

The theory section promised a number; here it is. `requires_grad` is the
flag that says "compute a gradient for this tensor", so summing over the
parameters that have it set counts precisely what the optimizer will
update. Read the percentage as the real claim of the whole method: that is
how much of the model we are allowed to move.

In [ ]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable: {trainable / 1e6:.2f}M")
print(f"total:     {total / 1e6:.2f}M")
print(f"training {100 * trainable / total:.2f}% of the model")

### Where the adapters actually went

The parameter count says *how much*; this says *where*. Each adapter is a
pair of thin matrices, and printing the shape of the first one per target
module shows the rank-16 dimension showing up in all seven places.

In [ ]:
adapter_shapes = {}
for name, module in model.named_modules():
    if name.endswith("lora_A.default"):
        target = name.split(".")[-3]
        adapter_shapes.setdefault(target, tuple(module.weight.shape))
for target, shape in sorted(adapter_shapes.items()):
    print(f"  {target:<12} lora_A {shape}")

### What evaluation needs

Evaluating a model on this task takes exactly two task-specific functions:
one that turns a puzzle into a chat prompt, and one that scores whatever
the model wrote. Bundling them means every evaluation call below fits on
one line.

In [ ]:
TASK = evaluation.FlatTask(compute_stats=compute_stats, to_chat_prompt=to_chat_prompt)

## 3. Baseline

A freshly initialized adapter changes nothing at all: `lora_B` starts at
zero, so the correction it adds is zero, and the wrapped model computes
exactly what the base model computed. That makes this a real check rather
than a formality — if this score does not match the FFT notebook's
base-model score, the adapters are wired in wrong.

In [ ]:
baseline = evaluation.evaluate_model(
    model, tokenizer, {"heldout": eval_splits["heldout"]}, f"{METHOD} (before training)", TASK,
    max_new_tokens=EVAL_MAX_NEW_TOKENS, batch_size=EVAL_BATCH_SIZE,
)
report.print_report(baseline)

## 4. Training

Identical schedule to the FFT notebook with one exception: the learning
rate is `LORA_LEARNING_RATE` (2e-4) rather than FFT's 5e-5. LoRA needs the
bigger step because it is only allowed to move a thin slice of the model,
so each step has less room to change the function. Keep `bf16=False,
fp16=False` here for the same reason as every other notebook — a Colab T4
cannot do bf16, and TRL's `bf16` default turns itself on when `fp16` is
left unset.

In [ ]:
train_dataset = train_rows.map(to_sft_format, remove_columns=train_rows.column_names)
eval_dataset = eval_splits["heldout"].map(
    to_sft_format, remove_columns=eval_splits["heldout"].column_names
)

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    max_steps=SFT_MAX_STEPS,
    learning_rate=LORA_LEARNING_RATE, weight_decay=0.01,
    warmup_ratio=0.05, lr_scheduler_type="cosine",
    per_device_train_batch_size=SFT_BATCH_SIZE,
    per_device_eval_batch_size=SFT_BATCH_SIZE,
    gradient_accumulation_steps=SFT_GRAD_ACCUM,
    max_length=MAX_SEQ_LENGTH,
    completion_only_loss=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="no",
    bf16=False,
    fp16=False,
    report_to="none",
    seed=SEED,
)
trainer = SFTTrainer(
    model=model, args=training_args, train_dataset=train_dataset,
    eval_dataset=eval_dataset, processing_class=tokenizer,
)
trainer.train()

`save_model` on a PEFT model writes only the adapter — much smaller than
the full checkpoint the FFT notebook produces.

In [ ]:
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
adapter_size = sum(p.stat().st_size for p in OUTPUT_DIR.glob("adapter_model*"))
print(f"saved to {OUTPUT_DIR}")
print(f"adapter size: {adapter_size / 1e6:.1f} MB")

### Merging the adapters back into the base model

The adapter saved above is a ~20 MB delta — cheap to store, but it needs
`peft` loaded to be used. The LoRA correction is just an added matrix
(`W + (alpha/r) BA`), so we can fold it back into the frozen weights once
and ship an ordinary model: no adapter layer at inference time, no `peft`
dependency, and the same function. This is the "costs nothing at inference
time" payoff from the theory section, made concrete.

`merge_and_unload` consumes the PEFT wrapper, so `model` from here on is a
plain `AutoModelForCausalLM` — the evaluation cells below run on it as-is,
and produce the same numbers the adapter would.

In [ ]:
# Fold the trained adapters into the frozen base weights. The correction
# W + (alpha/r) * B*A is just a matrix, so compute it once and ship an
# ordinary model — no PEFT wrapper, no adapter layers, no `peft` needed
# to load it afterwards.
model = model.merge_and_unload()
merged_dir = RESULTS_DIR / "lora_model_merged"
model.save_pretrained(str(merged_dir))
tokenizer.save_pretrained(str(merged_dir))

merged_size = sum(p.stat().st_size for p in merged_dir.glob("*.safetensors"))
print(f"merged model saved to {merged_dir}")
print(f"merged model size: {merged_size / 1e6:.1f} MB")
print(f"adapter size:      {adapter_size / 1e6:.1f} MB")

## 5. Training curves

The same caution as the FFT notebook: a low loss means the model
reproduces the demonstrations, broken moves included. Compare the shape of
this curve with FFT's — if the two look alike, the two methods really are
doing the same job.

In [ ]:
history = plotting.history_from_log(trainer.state.log_history)
plotting.plot_history(history, ["loss", "eval_loss"], RESULTS_DIR / "lora_loss.png",
                       title="LoRA: loss", ylabel="cross-entropy")

## 6. Evaluation

Score the adapted model on all four splits, then save the report to disk
so the GSPO notebook can pull it into the three-way comparison later.

In [ ]:
lora_report = evaluation.evaluate_model(model, tokenizer, eval_splits, METHOD, TASK,
                                         group_key=PROBE_GROUP_KEY,
                                         max_new_tokens=EVAL_MAX_NEW_TOKENS, batch_size=EVAL_BATCH_SIZE)
report.print_report(lora_report)
report.save_report(lora_report, RESULTS_DIR)

## 8. A single prediction

One 4-disk held-out puzzle in full, the same way the FFT notebook ends, so
you can put the two outputs side by side.

In [ ]:
heldout = eval_splits["heldout"]
index = next(i for i, r in enumerate(heldout) if r["n_disks"] == 4)
report.print_example(lora_report["splits"]["heldout"]["completions"][index], heldout[index],
                      compute_stats)